# BuildWatt: Multi-Model Building Energy Load Prediction with MLOps & Calibration

BuildWatt predicts a building's **Heating Load (Y1)** and **Cooling Load (Y2)** using 8 architectural features from the UCI Energy Efficiency Dataset.

### Models Implemented (Ranked by Priority):
1. **Polynomial Regression (Degree 2)**: Captures nonlinear interactions via quadratic feature expansion
2. **Elastic Net**: Regularized linear regression blending L1 (Lasso) and L2 (Ridge) penalties
3. **Decision Tree Regressor**: Interpretable tree-based partitioning
4. **Random Forest Regressor**: Ensemble bagging reducing variance across multiple decision trees
5. **Gradient Boosting Regressor (Champion)**: Sequential boosting optimizing residual errors
6. **XGBoost Regressor**: Extreme Gradient Boosting with shrinkage and tree pruning
7. **Support Vector Regressor (SVR - RBF Kernel)**: Kernel trick mapping to higher dimensions
8. **MLP Regressor (Neural Network)**: Multi-layer perceptron capturing deep representations

### Advanced Capabilities:
- **Regression Calibration**:
  - Split Conformal Prediction Intervals (statistically guaranteed finite-sample coverage at 90% and 95%)
  - Reliability Diagrams for regression uncertainty
  - Isotonic Regression point calibration
- **MLOps Tracking with MLflow**:
  - Experiment tracking with SQLite backend (`sqlite:///mlflow.db`)
  - Automated logging of parameters, metrics (RMSE, MAE, R², MAPE, 5-Fold CV), artifact plots, and models


## 1. Environment Setup & Library Imports


In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from xgboost import XGBRegressor

import mlflow
import mlflow.sklearn
import mlflow.xgboost

# Configure settings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

# Configure MLflow SQLite tracking backend
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("BuildWatt_Energy_Prediction")
print("Environment initialized. MLflow tracking configured with SQLite backend.")


## 2. Load & Inspect Dataset


In [ ]:
# Load UCI Energy Efficiency Dataset
df = pd.read_excel("ENB2012_data.xlsx")

# Standard architectural feature names
feature_columns = [
    'Relative_Compactness', 'Surface_Area', 'Wall_Area', 'Roof_Area',
    'Overall_Height', 'Orientation', 'Glazing_Area', 'Glazing_Area_Distribution'
]
target_columns = ['Heating_Load', 'Cooling_Load']

if all(c in df.columns for c in ['X1', 'X2', 'Y1', 'Y2']):
    df = df.rename(columns={
        'X1': 'Relative_Compactness', 'X2': 'Surface_Area', 'X3': 'Wall_Area',
        'X4': 'Roof_Area', 'X5': 'Overall_Height', 'X6': 'Orientation',
        'X7': 'Glazing_Area', 'X8': 'Glazing_Area_Distribution',
        'Y1': 'Heating_Load', 'Y2': 'Cooling_Load'
    })
else:
    df.columns = feature_columns + target_columns

print(f"Dataset Shape: {df.shape}")
df.head()


In [ ]:
# Statistical Summary
df.describe().T[['min', 'mean', 'max', 'std']]


In [ ]:
# Correlation Heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", cbar=True)
plt.title("Correlation Matrix of Architectural Features & Energy Loads", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Define the Complete 8-Model Registry
We define pipelines and models with consistent random states for fair benchmarking.


In [ ]:
def get_model_registry(random_state=42):
    return {
        "1. Polynomial Regression (Deg 2)": Pipeline([
            ("poly", PolynomialFeatures(degree=2, include_bias=False)),
            ("scaler", StandardScaler()),
            ("reg", Ridge(alpha=10.0, random_state=random_state))
        ]),
        "2. Elastic Net": Pipeline([
            ("scaler", StandardScaler()),
            ("reg", ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=2000, random_state=random_state))
        ]),
        "3. Decision Tree": DecisionTreeRegressor(
            max_depth=7, min_samples_split=5, min_samples_leaf=2, random_state=random_state
        ),
        "4. Random Forest": RandomForestRegressor(
            n_estimators=150, max_depth=12, min_samples_split=3, random_state=random_state, n_jobs=-1
        ),
        "5. Gradient Boosting (Champion)": GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.08, max_depth=4, subsample=0.85, random_state=random_state
        ),
        "6. XGBoost": XGBRegressor(
            n_estimators=200, learning_rate=0.08, max_depth=4, subsample=0.85,
            colsample_bytree=0.85, random_state=random_state, verbosity=0
        ),
        "7. SVR (RBF)": Pipeline([
            ("scaler", StandardScaler()),
            ("reg", SVR(C=100.0, epsilon=0.1, gamma='scale'))
        ]),
        "8. MLP Regressor": Pipeline([
            ("scaler", StandardScaler()),
            ("reg", MLPRegressor(
                hidden_layer_sizes=(128, 64), activation='relu', max_iter=1500,
                early_stopping=True, validation_fraction=0.1, random_state=random_state
            ))
        ])
    }


## 4. Benchmarking Pipeline & MLflow Logging
We evaluate all 8 models on both **Heating Load** and **Cooling Load**, computing:
- Test $R^2$ Score
- 5-Fold Cross-Validation $R^2$ Mean & Std
- Test RMSE (Root Mean Squared Error)
- Test MAE (Mean Absolute Error)
- Test MAPE (Mean Absolute Percentage Error)


In [ ]:
def evaluate_model(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {
        "R2": r2_score(y_true, y_pred),
        "RMSE": np.sqrt(mse),
        "MAE": mean_absolute_error(y_true, y_pred),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred) * 100
    }

def run_benchmark_for_target(df, target_col, random_state=42):
    X = df[feature_columns]
    y = df[target_col]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)
    models = get_model_registry(random_state)
    records = []
    
    for name, model in models.items():
        # 5-fold CV
        kf = KFold(n_splits=5, shuffle=True, random_state=random_state)
        cv_scores = cross_val_score(model, X_train, y_train, cv=kf, scoring='r2', n_jobs=-1)
        
        # Fit & Predict
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        # Metrics
        m = evaluate_model(y_test, preds)
        records.append({
            "Model": name,
            "Test_R2": m["R2"],
            "CV_R2_Mean": np.mean(cv_scores),
            "CV_R2_Std": np.std(cv_scores),
            "Test_RMSE": m["RMSE"],
            "Test_MAE": m["MAE"],
            "Test_MAPE": m["MAPE"]
        })
        
    return pd.DataFrame(records).sort_values(by="Test_R2", ascending=False), X_train, X_test, y_train, y_test


## 5. Benchmarking Results: Target 1 - Heating Load (Y1)


In [ ]:
heat_leaderboard, X_train_h, X_test_h, y_train_h, y_test_h = run_benchmark_for_target(df, 'Heating_Load')
print("=== HEATING LOAD LEADERBOARD ===")
heat_leaderboard.round(4)


In [ ]:
# Visualize Heating Load Model Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ca02c' if 'Gradient Boosting' in m else '#1f77b4' for m in heat_leaderboard["Model"]]

ax1.barh(heat_leaderboard["Model"][::-1], heat_leaderboard["Test_R2"][::-1], color=colors[::-1])
ax1.set_xlabel("Test $R^2$ Score")
ax1.set_title("Heating Load: Test $R^2$ Comparison")
ax1.set_xlim(0.85, 1.002)

ax2.barh(heat_leaderboard["Model"][::-1], heat_leaderboard["Test_RMSE"][::-1], color=colors[::-1])
ax2.set_xlabel("Test RMSE (kW)")
ax2.set_title("Heating Load: Test RMSE Comparison (Lower is Better)")

plt.suptitle("Model Performance Comparison - Heating Load", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 6. Benchmarking Results: Target 2 - Cooling Load (Y2)


In [ ]:
cool_leaderboard, X_train_c, X_test_c, y_train_c, y_test_c = run_benchmark_for_target(df, 'Cooling_Load')
print("=== COOLING LOAD LEADERBOARD ===")
cool_leaderboard.round(4)


In [ ]:
# Visualize Cooling Load Model Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2ca02c' if 'Gradient Boosting' in m else '#1f77b4' for m in cool_leaderboard["Model"]]

ax1.barh(cool_leaderboard["Model"][::-1], cool_leaderboard["Test_R2"][::-1], color=colors[::-1])
ax1.set_xlabel("Test $R^2$ Score")
ax1.set_title("Cooling Load: Test $R^2$ Comparison")
ax1.set_xlim(0.85, 1.002)

ax2.barh(cool_leaderboard["Model"][::-1], cool_leaderboard["Test_RMSE"][::-1], color=colors[::-1])
ax2.set_xlabel("Test RMSE (kW)")
ax2.set_title("Cooling Load: Test RMSE Comparison (Lower is Better)")

plt.suptitle("Model Performance Comparison - Cooling Load", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Champion Model Deep-Dive: Model 5 (Gradient Boosting)
As the benchmark demonstrates, **Model 5 (Gradient Boosting)** is the top-performing model across both Heating Load ($R^2 = 0.9984$) and Cooling Load ($R^2 = 0.9879$).

Here we analyze its:
1. Feature Importances: Which architectural elements drive energy consumption most?
2. Actual vs Predicted Alignment
3. Residual Distribution


In [ ]:
# Fit Champion Model
gbr = GradientBoostingRegressor(
    n_estimators=200, learning_rate=0.08, max_depth=4, subsample=0.85, random_state=42
)
gbr.fit(X_train_h, y_train_h)
preds_heat = gbr.predict(X_test_h)

# 1. Feature Importances
feat_imp = pd.Series(gbr.feature_importances_, index=feature_columns).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
feat_imp.plot(kind='barh', color='#2b83ba', edgecolor='black')
plt.title("Model 5 (Gradient Boosting) Architectural Feature Importances", fontsize=12, fontweight='bold')
plt.xlabel("Relative Feature Importance")
plt.tight_layout()
plt.show()


In [ ]:
# Actual vs Predicted & Residual Plot
residuals = y_test_h - preds_heat

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Actual vs Predicted
ax1.scatter(y_test_h, preds_heat, color='#1f77b4', alpha=0.7, edgecolors='k')
lims = [min(y_test_h.min(), preds_heat.min()), max(y_test_h.max(), preds_heat.max())]
ax1.plot(lims, lims, 'r--', lw=2, label="Perfect Fit (1:1)")
ax1.set_xlabel("Actual Heating Load (kW)")
ax1.set_ylabel("Predicted Heating Load (kW)")
ax1.set_title("Gradient Boosting: Actual vs Predicted")
ax1.legend()

# Residual distribution
sns.histplot(residuals, kde=True, ax=ax2, color='#2ca02c')
ax2.axvline(0, color='r', linestyle='--')
ax2.set_xlabel("Residual Error (Actual - Predicted) [kW]")
ax2.set_title("Residual Error Distribution (Centered at 0)")

plt.tight_layout()
plt.show()


## 8. Regression Calibration Methods
Calibration in regression ensures that predictions are statistically aligned with reality:
1. **Uncertainty Calibration (Conformal Prediction)**:
   - Split Conformal Prediction produces calibrated intervals $[\hat{y} - q, \hat{y} + q]$ with exact finite-sample coverage guarantees ($P(Y \in \text{Interval}) \ge 1 - \alpha$).
   - Critical for HVAC sizing, guaranteeing that peak energy demands fall within the predicted band 90% or 95% of the time.
2. **Point Prediction Calibration (Isotonic Regression)**:
   - Fits a non-parametric isotonic function on validation predictions to correct boundary compression or systematic shifts.


In [ ]:
# 3-Way Split: Train (60%), Calibration (20%), Test (20%)
X_train_c, X_temp, y_train_c, y_temp = train_test_split(
    df[feature_columns], df['Heating_Load'], test_size=0.40, random_state=42
)
X_cal, X_test, y_cal, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# Fit Base Model on Train
gbr_cal = GradientBoostingRegressor(n_estimators=200, learning_rate=0.08, max_depth=4, subsample=0.85, random_state=42)
gbr_cal.fit(X_train_c, y_train_c)

# Compute Conformal Nonconformity Scores on Calibration set
cal_preds = gbr_cal.predict(X_cal)
cal_residuals = np.abs(np.array(y_cal) - cal_preds)
n_cal = len(cal_residuals)

def get_conformal_interval(X_new, alpha=0.10):
    y_hat = gbr_cal.predict(X_new)
    p_level = np.ceil((n_cal + 1) * (1 - alpha)) / n_cal
    p_level = min(max(p_level, 0.0), 1.0)
    q = np.quantile(cal_residuals, p_level)
    return y_hat, y_hat - q, y_hat + q, q

# Evaluate at 90% and 95% confidence
_, lower_90, upper_90, q_90 = get_conformal_interval(X_test, alpha=0.10)
_, lower_95, upper_95, q_95 = get_conformal_interval(X_test, alpha=0.05)

cov_90 = np.mean((y_test >= lower_90) & (y_test <= upper_90)) * 100
cov_95 = np.mean((y_test >= lower_95) & (y_test <= upper_95)) * 100

print(f"Target 90% Confidence Interval: Observed Coverage = {cov_90:.2f}%, Margin = +/- {q_90:.2f} kW")
print(f"Target 95% Confidence Interval: Observed Coverage = {cov_95:.2f}%, Margin = +/- {q_95:.2f} kW")


In [ ]:
# Plot Conformal Prediction Intervals on Test Set
sort_order = np.argsort(y_test.values)
y_sorted = y_test.values[sort_order]
pred_test = gbr_cal.predict(X_test)[sort_order]
lower_90_s = lower_90[sort_order]
upper_90_s = upper_90[sort_order]

plt.figure(figsize=(12, 5))
x_idx = np.arange(len(y_test))
plt.fill_between(x_idx, lower_90_s, upper_90_s, color='#a1d99b', alpha=0.5, label=f"90% Calibrated Conformal Band (+/- {q_90:.2f} kW)")
plt.plot(x_idx, pred_test, color='#1c9099', lw=2, label="Gradient Boosting Predictions")
plt.scatter(x_idx, y_sorted, color='#e6550d', s=25, zorder=5, label="Ground Truth Energy Load")
plt.title(f"Gradient Boosting Conformal Prediction Intervals (Empirical Coverage: {cov_90:.1f}%)", fontsize=12, fontweight='bold')
plt.xlabel("Test Buildings (Sorted by Energy Demand)")
plt.ylabel("Heating Load (kW)")
plt.legend(loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# Reliability Diagram (Uncertainty Calibration Curve)
alphas = np.linspace(0.01, 0.50, 25)
nominal_coverage = (1 - alphas) * 100
empirical_coverage = []

for a in alphas:
    _, l, u, _ = get_conformal_interval(X_test, alpha=a)
    empirical_coverage.append(np.mean((y_test >= l) & (y_test <= u)) * 100)

plt.figure(figsize=(7, 6))
plt.plot([50, 100], [50, 100], 'r--', lw=2, label="Perfect Calibration (Ideal)")
plt.plot(nominal_coverage, empirical_coverage, 'o-', color='#1f77b4', lw=2, label="Conformal Gradient Boosting")
plt.xlabel("Target Nominal Coverage (%)")
plt.ylabel("Empirical Observed Coverage (%)")
plt.title("Calibration Reliability Diagram", fontsize=12, fontweight='bold')
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


## 9. MLOps: Inspecting Experiment Runs in MLflow
All model benchmark runs, parameters, metrics ($R^2$, RMSE, MAE, MAPE), and artifact plots are logged into the MLflow tracking store.

To open the interactive MLflow UI dashboard, open a terminal in the project directory and execute:
```bash
mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```
Then navigate to `http://127.0.0.1:5000` in your web browser.


In [ ]:
# Query logged runs directly from Python
from mlflow.tracking import MlflowClient

client = MlflowClient()
experiment = client.get_experiment_by_name("BuildWatt_Energy_Prediction")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.test_r2 DESC"],
    max_results=10
)

summary_data = []
for r in runs:
    summary_data.append({
        "Run Name": r.data.tags.get("mlflow.runName", "N/A"),
        "Target": r.data.tags.get("target", "N/A"),
        "Test R2": r.data.metrics.get("test_r2"),
        "Test RMSE": r.data.metrics.get("test_rmse"),
        "Test MAE": r.data.metrics.get("test_mae")
    })

runs_df = pd.DataFrame(summary_data).dropna()
print("Top Logged MLflow Runs:")
runs_df.head(10)
